# Sobol sensitivity analysis using saved fits to the data

Program to read in the zarr collection holding the fits to compute sobol indices

The FFDI code needed 3-23-.10 but this was not tested using this code.




#### required packages

In [ ]:
import intake
import xarray as xr
from matplotlib import pyplot as plt
import numpy as np

from glob import glob
import pathlib
import traceback
from datetime import datetime

from xclim.indices import (
    keetch_byram_drought_index,
    griffiths_drought_factor,
    mcarthur_forest_fire_danger_index
)

%load_ext autoreload
%autoreload 2


# importing sys
import sys
# adding plotting module to the system path
sys.path.insert(0, '/g/data/xv83/rxm599/acs/plotting_maps')
# import ACS plotting maps and Xarray.
from acs_plotting_maps import *
from acs_area_statistics import acs_regional_stats, get_regions

import geopandas as gpd
import pandas as pd
import regionmask

sys.path.insert(0, '/g/data/xv83/rxm599/acs/hazard_fire/paper2')
import sa as sa
import df as df


#### start a local Dask client

In [ ]:
from dask.distributed import Client
import dask

# Set configuration options
dask.config.set({
    'distributed.comm.timeouts.connect': '90s',  # Timeout for connecting to a worker
    'distributed.comm.timeouts.tcp': '90s',  # Timeout for TCP communications
})

#cluster = LocalCluster(
#    n_workers=28,          # Number of workers
#    threads_per_worker=1 #Threads per worker
#    #memory_limit='8GB' # Memory limit per each worker commented out
#)
#client = Client(cluster)

client = Client()
client

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Set parameters

In [ ]:
# parameters
# 0,2,3 failed
mindex=1
#lon1=140; lon2=150 
#lat1=-45; lat2= -32
#lat1=-45 lat2= -40 lon1=144 lon2=149
#lon1=145 ; lon2=146
#lat1=-37 ; lat2=-36
lat1=-45
lat2= -10
lon1=110
lon2=155
p_ext=0.95
p_ext=0.99

t1='2015-01-01'
t2='2035-01-01'
dirstore='sa_2020'


In [ ]:
# extra info
mindexp100=mindex+100
mobs=39

## Obtain the desired catalogues of the simulations to processe

In [ ]:
dtype='/g/data/ia39/ncra/fire/bias-output/'
zarr_path=dtype+ 'zarr/*ssp370*'
ffdi_path=dtype+'ffdi/*ssp370*FFDI*'
kbdi_path=dtype+'ffdi/*ssp370*KBDI*'
# observations 
zobs='/g/data/ia39/ncra/fire/bias-input/zarr/*ERA*' 
fobs='/g/data/ia39/ncra/fire/bias-input/ffdi/*ERA*FFDI*'
kobs='/g/data/ia39/ncra/fire/bias-input/ffdi/*ERA*KBDI*'

mRuns = sorted(glob(zarr_path)) + sorted(glob(zobs))
mFFDI = sorted(glob(ffdi_path))+ sorted(glob(fobs))
mKBDI = sorted(glob(kbdi_path)) + sorted(glob(kobs))
print(len(mRuns))
print(len(mFFDI))

# print code to ensure the files match
ifile=-1
for file in mRuns: 
    ifile=ifile+1
#    print(ifile, file)
ifile=-1
for file in mFFDI: 
    ifile=ifile+1
#    print(ifile, file)

# Process one ensemble 

In [ ]:
# From one catalogue list save variables
ds0=xr.open_zarr(mRuns[mindex])
df0=xr.open_zarr(mFFDI[mindex])
dk0=xr.open_zarr(mKBDI[mindex])

print(mRuns[mindex])
print(mFFDI[mindex])
print(mKBDI[mindex])


# compute DF 
# kbdi has a 20 extra points at the start for DF calculation
if mindex != mobs: 
# for projections    
    pra = ds0.prAdjust.sel(time=slice(ds0.prAdjust.time[0],'2099-12-31'))
    kbdi=dk0.KBDI.sel(time=slice(pra.time[0],pra.time[-1] ))
    DF = df.griffiths_drought_factor_dask_exact(pra, kbdi)
else:
    print("obs")
    pra = ds0.pr.sel(time=slice(ds0.pr.time[0],'2099-12-31'))
    kbdi=dk0.KBDI.sel(time=slice(pra.time[0],pra.time[-1] ))
    DF = df.griffiths_drought_factor_dask_exact(pra, kbdi)
    
print(pra)
print(kbdi)


In [ ]:
# compute the 95% value from first 20 years
if mindex != mobs:
    tr1='2015-01-01'; tr2='2035-01-01'
else:
    tr1='2000-01-01'; tr2='2020-01-01'
    t1=tr1; t2=tr2
#    tr1='2003-01-01'; tr2='2023-12-31'

df1=df0.sel(time=slice(tr1,tr2))
d95a=df1.FFDI.quantile(p_ext,dim='time')


## mask info

## Read in fit results

In [ ]:
%%time
ds1=0; ds2=0
outf1='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_fits.zarr'
print(mindex)
print(outf1)

ds1 = xr.open_zarr(outf1)

In [ ]:
%%time
tfit=ds1.tfit.load()
hfit=ds1.hfit.load()
wfit=ds1.wfit.load()
dfit=ds1.dfit.load()

In [ ]:
lev1=np.arange(0, 1, .1)
lev1 = np.linspace(0, .0001, 11)
plt.figure(figsize=(8, 8))
plt.subplot(2,2,1); tfit.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("Tmax")
plt.subplot(2,2,2); hfit.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("Hmin")
plt.subplot(2,2,3); dfit.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("DI")
plt.subplot(2,2,4); wfit.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("Wmax")

In [ ]:
%%time

def bad(a11): # Find locations where condition is True
    locations = a11 < .0000002
# Stack dimensions into a flat index
    stacked = locations.stack(all_points=("lon", "lat"))
# Get the coordinates where the condition is True
    matching_indices = stacked.where(stacked, drop=True)
# Print the matching indices
#    print(matching_indices.coords)
    count = (matching_indices).sum().item()  # .item() to get Python scalar
    print('number of points=',count)
    return matching_indices

res=bad(tfit.sel(parameter1='p-value'))
res=bad(hfit.sel(parameter1='p-value'))
res=bad(dfit.sel(parameter1='p-value'))
res=bad(wfit.sel(parameter1='p-value'))


In [ ]:
ds1.Wmax_min.plot()

### Do it again with lognorm fits

In [ ]:
%%time
outf2='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_lfits.zarr'
print(mindex)
print(outf1)

ds2 = xr.open_zarr(outf2)

In [ ]:
%%time
tfit1=ds2.tfit.load()
hfit1=ds2.hfit.load()
wfit1=ds2.wfit.load()
dfit1=ds2.dfit.load()

In [ ]:
lev1=np.arange(0, 1, .1)
lev1 = np.linspace(0, .0001, 11)
plt.figure(figsize=(8, 8))
plt.subplot(2,2,1); tfit1.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("Tmax")
plt.subplot(2,2,2); hfit1.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("Hmin")
plt.subplot(2,2,3); dfit1.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("DI")
plt.subplot(2,2,4); wfit1.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("Wmax")

In [ ]:
%%time

res=bad(tfit1.sel(parameter1='p-value'))
res=bad(hfit1.sel(parameter1='p-value'))
res=bad(dfit1.sel(parameter1='p-value'))
res=bad(wfit1.sel(parameter1='p-value'))


In [ ]:
a1=(tfit.sel(parameter1='p-value') - tfit1.sel(parameter1='p-value'))
#a1.max().values
la=-30;lo=115.05
print(tfit.sel(lon=lo,lat=la).values)
print(tfit1.sel(lon=lo,lat=la).values)

In [ ]:
a1=wfit.sel(parameter1='p-value')
a2=wfit1.sel(parameter1='p-value')
b=xr.apply_ufunc(np.maximum, a1, a2)
res=bad(b)
a1=tfit.sel(parameter1='p-value')
a2=tfit1.sel(parameter1='p-value')
b=xr.apply_ufunc(np.maximum, a1, a2)
res=bad(b)
a1=hfit.sel(parameter1='p-value')
a2=hfit1.sel(parameter1='p-value')
b=xr.apply_ufunc(np.maximum, a1, a2)
res=bad(b)
a1=dfit.sel(parameter1='p-value')
a2=dfit1.sel(parameter1='p-value')
b=xr.apply_ufunc(np.maximum, a1, a2)
res=bad(b)

In [ ]:
#sys.exit(1)

## Sobol Indices

In [ ]:
%%time
outf1='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_fits.zarr'
print(mindex)
print(outf1)

ds1 = xr.open_zarr(outf1)
ds1

In [ ]:
%%time
tfit=ds1.tfit.load()
hfit=ds1.hfit.load()
wfit=ds1.wfit.load()
dfit=ds1.dfit.load()

t_min=ds1.Tmax_min
h_min=ds1.Hmin_min
w_min=ds1.Wmax_min
d_min=ds1.DI_min

In [ ]:
la=-30;lo=115.05
tfit1=tfit.sel(lat=la,lon=lo).load()
wfit1=wfit.sel(lat=la,lon=lo).load()
dfit1=dfit.sel(lat=la,lon=lo).load()
hfit1=hfit.sel(lat=la,lon=lo).load()

t_min1=t_min.sel(lat=la,lon=lo)#.load()
h_min1=h_min.sel(lat=la,lon=lo)#.load()
w_min1=w_min.sel(lat=la,lon=lo)#.load()
d_min1=d_min.sel(lat=la,lon=lo)#.load()


In [ ]:
%%time
sa1,sa2=sa.xr_sobol_indices(tfit1,hfit1,wfit1,dfit1)
print(sa1.values)
print(sa2.values)

sa1=sa.xr_sobol_indices_v2(dfit1,tfit1,hfit1,wfit1,d_min1*0,t_min1*0,h_min1*0,w_min1*0,4096*4)
print(sa1.values)

sa1=sa.xr_sobol_indices_v2(dfit1,tfit1,hfit1,wfit1,d_min1,t_min1,h_min1,w_min1,4096*4)
print(sa1.values)


In [ ]:
%%time
#sa1,sa2=sa.xr_sobol_indices(tfit,hfit,wfit,dfit)
#sa1,sa2=sa.xr_sobol_indices_v2(dfit,tfit,hfit,wfit,d_min,t_min,h_min,w_min,4096*4)
sa1=sa.xr_sobol_indices_v2(dfit,tfit,hfit,wfit,d_min,t_min,h_min,w_min,4096*4)
sa1

In [ ]:
%%time
sa1.load()
lev = np.linspace(0, 1, 21)
lev = np.linspace(0, .5, 21)
plt.figure(figsize=(8, 10))
plt.subplot(4,2,1); sa1.sel(sa_1='DI',stat=0).plot(levels=lev,cmap="coolwarm")
plt.title("DI")
plt.subplot(4,2,2); sa1.sel(sa_1='DI',stat=1).plot(levels=lev,cmap="coolwarm")
plt.title("DI")
plt.subplot(4,2,3); sa1.sel(sa_1='Tmax',stat=0).plot(levels=lev,cmap="coolwarm")
plt.title("Tmax")
plt.subplot(4,2,4); sa1.sel(sa_1='Tmax',stat=1).plot(levels=lev,cmap="coolwarm")
plt.title("Tmax")
plt.subplot(4,2,5); sa1.sel(sa_1='Hmin',stat=0).plot(levels=lev,cmap="coolwarm")
plt.title("Hmin")
plt.subplot(4,2,6); sa1.sel(sa_1='Hmin',stat=1).plot(levels=lev,cmap="coolwarm")
plt.title("Hmin")
plt.subplot(4,2,7); sa1.sel(sa_1='Wmax',stat=0).plot(levels=lev,cmap="coolwarm")
plt.title("Wmax")
plt.subplot(4,2,8); sa1.sel(sa_1='Wmax',stat=1).plot(levels=lev,cmap="coolwarm")
plt.title("Wmax")


In [ ]:
%%time
outf1='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_sa.nc'
print(mindex)
print(outf1)
sa1.name='SA'
#ds = xr.merge([sa1,sa2])  #,dfit,dfitb])

sa1.to_netcdf(outf1,mode='w')


In [ ]:
#client.shutdown()